In [1]:
import os
import json
import time
from openai import OpenAI

API_KEY = "sk-proj-xxxxxxxxxxxxxx" 
client = OpenAI(api_key=API_KEY)

In [2]:
def generate_gold_label(model, system_prompt, user_content):
    """Helper for generating strict answer keys"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0 # Strict deterministic output
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"  [!] API Error: {e}")
        return None

In [3]:
# API connection test
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say hello in one word"
)

print(response.output_text)

Hello!


In [4]:
# AMI (Test Set - Gold Labels)
INPUT_FILE = "phase1_ami_clean_test.jsonl"
OUTPUT_FILE = "phase1b_ami_test_gold.jsonl"
MODEL = "gpt-4.1-mini"

SYSTEM_PROMPT = """You are an expert AI data annotator for a machine learning pipeline.
Task: Analyze the meeting transcript and extract the key information in a highly structured format. 
You MUST strictly format your response using the EXACT structure below. Do not add any conversational text before or after.

Thought: Key speakers: [List the speaker letters, e.g., A, B, C]
Main decisions: [List the main decisions as concise points]
Action items: [List actionable items and who is doing them, if mentioned]
Summary: [Write a concise abstractive summary of 1 to 2 sentences maximum]"""

In [5]:
print(f"Generating Gold Labels for {INPUT_FILE}...")

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f_in, open(OUTPUT_FILE, 'w') as f_out:
        lines = f_in.readlines()
        print(f"  -> Found {len(lines)} test samples.")
        
        for i, line in enumerate(lines):
            data = json.loads(line)
            
            if data.get('output') == "TO_BE_GENERATED":
                output = generate_gold_label(MODEL, SYSTEM_PROMPT, data['input'])
                if output:
                    data['output'] = output
            
            f_out.write(json.dumps(data) + "\n")
            
            if (i+1) % 5 == 0: 
                print(f"  [AMI Test] Processed {i+1}/{len(lines)}")

    print(f"AMI Test Key saved to {OUTPUT_FILE}")
else:
    print(f"Error: {INPUT_FILE} not found.")

Generating Gold Labels for phase1_ami_clean_test.jsonl...
  -> Found 26 test samples.
  [AMI Test] Processed 5/26
  [AMI Test] Processed 10/26
  [AMI Test] Processed 15/26
  [AMI Test] Processed 20/26
  [AMI Test] Processed 25/26
AMI Test Key saved to phase1b_ami_test_gold.jsonl
